In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("GEMINI_API_KEY")
secret_value_1 = user_secrets.get_secret("HF_TOKEN")

In [3]:
# ============================================================
# CELL 0 — Environment Check + Disk Audit
# ============================================================
import os, shutil, torch

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

for path in ["/kaggle/working", "/tmp"]:
    total, used, free = shutil.disk_usage(path)
    print(f"{path}")
    print(f"  Total: {total/1e9:.1f} GB | Used: {used/1e9:.1f} GB | Free: {free/1e9:.1f} GB")

print(f"\nGPU:  {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"CUDA: {torch.version.cuda}")

/kaggle/working
  Total: 21.0 GB | Used: 0.0 GB | Free: 20.9 GB
/tmp
  Total: 8656.9 GB | Used: 7346.4 GB | Free: 1310.5 GB

GPU:  Tesla T4
VRAM: 15.6 GB
CUDA: 12.8


In [4]:
# ============================================================
# CELL 1 — Install
# ============================================================
import subprocess
subprocess.run(["pip", "install", "unsloth[colab-new]", "unsloth_zoo", "--quiet"])
subprocess.run(["pip", "install", "--no-deps", "trl", "peft",
                "accelerate", "bitsandbytes", "--quiet"])
print("✓ Done")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 2.9 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 104.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.4/67.4 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2025.9.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2025.9.0 which is incompatible.


✓ Done


In [5]:
# ============================================================
# CELL 2 — Load Gemma 4 E2B
# Cache model weights to /tmp to avoid /kaggle/working usage
# ============================================================
from unsloth import FastModel
import torch, os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Route HuggingFace cache to /tmp — model weights won't touch /kaggle/working
os.environ["HF_HOME"]            = "/tmp/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/tmp/hf_cache"
os.environ["HF_DATASETS_CACHE"]  = "/tmp/hf_datasets_cache"

os.makedirs("/tmp/hf_cache", exist_ok=True)
os.makedirs("/tmp/hf_datasets_cache", exist_ok=True)
os.makedirs("/tmp/pathos", exist_ok=True)

model, tokenizer = FastModel.from_pretrained(
    model_name      = "unsloth/gemma-4-e2b-it",
    max_seq_length  = 1024,
    load_in_4bit    = True,
    full_finetuning = False,
)

model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = True,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r            = 16,
    lora_alpha   = 16,
    lora_dropout = 0,
    bias         = "none",
    random_state = 42,
)

total = torch.cuda.get_device_properties(0).total_memory/1e9
alloc = torch.cuda.memory_allocated()/1e9
print(f"VRAM allocated: {alloc:.2f} GB")
print(f"VRAM free:      {total-alloc:.2f} GB")
model.print_trainable_parameters()

# Confirm disk usage
for path in ["/kaggle/working", "/tmp"]:
    _, used, free = shutil.disk_usage(path)
    print(f"{path} — Used: {used/1e9:.1f} GB | Free: {free/1e9:.1f} GB")

print("✓ Model ready")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/203 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

VRAM allocated: 8.26 GB
VRAM free:      7.38 GB
trainable params: 29,859,840 || all params: 5,153,037,856 || trainable%: 0.5795
/kaggle/working — Used: 0.0 GB | Free: 20.9 GB
/tmp — Used: 7355.3 GB | Free: 1301.6 GB
✓ Model ready


In [6]:
# ============================================================
# CELL 3 — Build Unified PathOS Dataset
# Dataset cache also routed to /tmp
# ============================================================
from datasets import load_dataset
from torch.utils.data import Dataset, ConcatDataset
from PIL import Image
from collections import defaultdict
import random, torch, os, shutil

random.seed(42)

# ── Shared config ────────────────────────────────────────────

SYSTEM_PROMPT = """You are PathOS, an AI pathologist assistant for histopathology analysis.
You assist pathology labs by analyzing digitized tissue slides with expert-level precision.
Your role:
- Identify tissue types and cellular components from H&E stained images
- Describe morphological features with clinical accuracy
- Generate structured pathology reports for lab workflows
- Answer diagnostic questions to support pathologist review
Always lead with a direct, specific answer grounded in visible histological features."""

REPORT_TEMPLATE = """PATHOS LAB REPORT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Specimen:         Histopathology patch
Tissue Type:      {tissue}
Primary Finding:  {finding}
Morphology:       {morphology}
Clinical Imp.:    {clinical}
Confidence:       {confidence}
Pathologist Note: {note}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━"""

LABEL_MAP = {
    0:"ADI", 1:"BACK", 2:"DEB", 3:"LYM",
    4:"MUC", 5:"MUS",  6:"NORM",7:"STR", 8:"TUM"
}

TISSUE_KB = {
    "TUM": {
        "name":     "colorectal adenocarcinoma",
        "morph":    "irregular glands, nuclear pleomorphism, hyperchromatic nuclei, mitotic figures",
        "clinical": "malignant; requires oncology staging workup",
        "finding":  "Malignant glandular epithelium identified",
        "conf":     "High",
        "note":     "Correlate with TNM staging and molecular markers"
    },
    "LYM": {
        "name":     "lymphocytic infiltrate",
        "morph":    "dense small round cells, condensed nuclei, scant cytoplasm, sheet-like arrangement",
        "clinical": "immune response or lymphocytic colitis; exclude lymphoma",
        "finding":  "Dense lymphocytic aggregates in lamina propria",
        "conf":     "High",
        "note":     "Clinical correlation with endoscopy and CBC recommended"
    },
    "STR": {
        "name":     "cancer-associated stroma",
        "morph":    "desmoplastic spindle fibroblasts, dense collagen, myofibroblastic differentiation",
        "clinical": "tumor microenvironment; associated with invasion and poor prognosis",
        "finding":  "Desmoplastic stromal reaction adjacent to tumor",
        "conf":     "Moderate-High",
        "note":     "Evaluate with adjacent tumor sections for invasion depth"
    },
    "ADI": {
        "name":     "adipose tissue",
        "morph":    "univacuolated lipid droplets, peripheral nuclei, thin fibrous septa",
        "clinical": "pericolonic fat; assess for tumor infiltration",
        "finding":  "Normal mature adipose tissue",
        "conf":     "High",
        "note":     "Document proximity to tumor front for pT staging"
    },
    "MUC": {
        "name":     "mucinous component",
        "morph":    "extracellular mucin pools, floating epithelial clusters, pale acellular H&E material",
        "clinical": "mucinous adenocarcinoma subtype if tumor-associated; distinct prognosis",
        "finding":  "Abundant extracellular mucin with epithelial clusters",
        "conf":     "Moderate",
        "note":     "Full section review; mucinous subtype requires separate staging"
    },
    "MUS": {
        "name":     "smooth muscle (muscularis)",
        "morph":    "elongated spindle cells, cigar-shaped nuclei, intersecting fascicles, eosinophilic cytoplasm",
        "clinical": "muscularis propria landmark critical for invasion staging",
        "finding":  "Normal smooth muscle bundles of muscularis layer",
        "conf":     "High",
        "note":     "Key landmark for pT2 vs pT3 distinction"
    },
    "NORM": {
        "name":     "normal colon mucosa",
        "morph":    "regular crypts, columnar epithelium, goblet cells, intact basement membrane",
        "clinical": "no pathological abnormality",
        "finding":  "Normal colonic mucosal architecture preserved",
        "conf":     "High",
        "note":     "No further pathological workup required for this patch"
    },
    "DEB": {
        "name":     "necrotic debris",
        "morph":    "acellular eosinophilic material, ghost outlines, karyolysis, karyorrhexis",
        "clinical": "tumor necrosis; common in high-grade rapidly proliferating tumors",
        "finding":  "Necrotic tumor debris with inflammatory infiltrate",
        "conf":     "Moderate",
        "note":     "Quantify necrosis extent; correlate with tumor grade"
    },
    "BACK": {
        "name":     "background (non-tissue)",
        "morph":    "glass slide background, no cellular elements, processing artifacts",
        "clinical": "non-diagnostic",
        "finding":  "No tissue present in this image region",
        "conf":     "N/A",
        "note":     "Re-examine adjacent patches for diagnostic tissue"
    },
}

NCT_QUESTIONS = [
    "What tissue type is present in this histopathology patch?",
    "Identify the tissue class in this H&E stained section.",
    "What are the key histological features visible here?",
    "Describe the pathological finding in this tissue section.",
    "What does this digitized slide patch show?",
    "Classify the tissue type and describe its features.",
]

REPORT_QUESTIONS = [
    "Generate a PathOS lab report for this histopathology patch.",
    "Produce a structured pathology report for this digitized slide.",
    "Analyze this tissue patch and generate a formal lab report.",
]

YN_FORMAT   = "{ans}. The histological features in this image {support} this finding."
OPEN_FORMAT = "{ans}. This is identifiable from the cellular morphology and tissue architecture visible."

def make_nct_answer(label_key):
    t = TISSUE_KB[label_key]
    return (
        f"This patch shows {t['name']}. "
        f"Key morphological features: {t['morph']}. "
        f"Clinical significance: {t['clinical']}."
    )

def make_report(label_key):
    t = TISSUE_KB[label_key]
    return REPORT_TEMPLATE.format(
        tissue    = t['name'],
        finding   = t['finding'],
        morphology= t['morph'],
        clinical  = t['clinical'],
        confidence= t['conf'],
        note      = t['note'],
    )

def make_message(img, user_text, assistant_text):
    return [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": img},
                {"type": "text",  "text": user_text}
            ]
        },
        {
            "role": "assistant",
            "content": [{"type": "text", "text": assistant_text}]
        }
    ]

def tokenize(messages, tok):
    # extract image from messages
    img  = messages[0]["content"][0]["image"]
    text = tok.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    inputs = tok(
        text=text, images=[img],
        return_tensors="pt", padding=True,
        truncation=True, max_length=1024,
    )
    return {k: v.squeeze(0) for k, v in inputs.items()}

# ── Load NCT-CRC (cached to /tmp) ───────────────────────────
print("Loading NCT-CRC-HE-100K...")
nct = load_dataset(
    "1aurent/NCT-CRC-HE",
    split      = "NCT_CRC_HE_100K",
    cache_dir  = "/tmp/hf_datasets_cache",
)
print(f"NCT-CRC loaded: {len(nct)} samples")

class_buckets = defaultdict(list)
for i, sample in enumerate(nct):
    key = LABEL_MAP[sample['label']]
    class_buckets[key].append(i)

qa_pairs, report_pairs = [], []
for label, indices in class_buckets.items():
    random.shuffle(indices)
    qa_pairs.extend(    [(i, label) for i in indices[:200]])
    report_pairs.extend([(i, label) for i in indices[200:250]])

random.shuffle(qa_pairs)
random.shuffle(report_pairs)
print(f"NCT QA: {len(qa_pairs)} | NCT Reports: {len(report_pairs)}")

# ── Load PathVQA (cached to /tmp) ───────────────────────────
print("\nLoading PathVQA...")
pvqa = load_dataset(
    "flaviagiammarino/path-vqa",
    cache_dir = "/tmp/hf_datasets_cache",
)
train_pvqa = pvqa['train']

def is_quality(s):
    a = s['answer'].strip().lower()
    return len(a) >= 2 and a not in ['n/a','none','unknown']

all_idx   = [i for i in range(len(train_pvqa)) if is_quality(train_pvqa[i])]
yn_idx    = [i for i in all_idx if train_pvqa[i]['answer'].lower() in ['yes','no']]
open_idx  = [i for i in all_idx if train_pvqa[i]['answer'].lower() not in ['yes','no']]

random.shuffle(yn_idx)
random.shuffle(open_idx)

pvqa_selected = yn_idx[:1500] + open_idx[:1500]
random.shuffle(pvqa_selected)
print(f"PathVQA: {len(pvqa_selected)} samples (1500 yn + 1500 open)")

# ── Dataset classes ──────────────────────────────────────────

class NCTQADataset(Dataset):
    def __init__(self, dataset, pairs, tokenizer):
        self.dataset, self.pairs, self.tok = dataset, pairs, tokenizer
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        data_idx, label = self.pairs[idx]
        img = self.dataset[data_idx]['image']
        if img.mode != 'RGB': img = img.convert('RGB')
        msgs = make_message(
            img,
            f"{SYSTEM_PROMPT}\n\nQuestion: {random.choice(NCT_QUESTIONS)}",
            make_nct_answer(label)
        )
        return tokenize(msgs, self.tok)

class NCTReportDataset(Dataset):
    def __init__(self, dataset, pairs, tokenizer):
        self.dataset, self.pairs, self.tok = dataset, pairs, tokenizer
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        data_idx, label = self.pairs[idx]
        img = self.dataset[data_idx]['image']
        if img.mode != 'RGB': img = img.convert('RGB')
        msgs = make_message(
            img,
            f"{SYSTEM_PROMPT}\n\n{random.choice(REPORT_QUESTIONS)}",
            make_report(label)
        )
        return tokenize(msgs, self.tok)

class PathVQADataset(Dataset):
    def __init__(self, dataset, indices, tokenizer):
        self.dataset, self.indices, self.tok = dataset, indices, tokenizer
    def __len__(self): return len(self.indices)
    def __getitem__(self, idx):
        sample  = self.dataset[self.indices[idx]]
        answer  = sample['answer'].strip()
        is_yn   = answer.lower() in ['yes','no']
        img     = sample['image']
        if img.mode != 'RGB': img = img.convert('RGB')
        img     = img.resize((224, 224))
        if is_yn:
            support  = "confirm" if answer.lower()=="yes" else "do not support"
            ans_text = YN_FORMAT.format(ans=answer.capitalize(), support=support)
        else:
            ans_text = OPEN_FORMAT.format(ans=answer.capitalize())
        msgs = make_message(
            img,
            f"{SYSTEM_PROMPT}\n\nQuestion: {sample['question']}",
            ans_text
        )
        return tokenize(msgs, self.tok)

nct_qa_ds     = NCTQADataset(nct, qa_pairs, tokenizer)
nct_report_ds = NCTReportDataset(nct, report_pairs, tokenizer)
pvqa_ds       = PathVQADataset(train_pvqa, pvqa_selected, tokenizer)
combined_ds   = ConcatDataset([nct_qa_ds, nct_report_ds, pvqa_ds])

print(f"\nDataset composition:")
print(f"  NCT-CRC QA:     {len(nct_qa_ds)}")
print(f"  NCT-CRC Reports:{len(nct_report_ds)}")
print(f"  PathVQA:        {len(pvqa_ds)}")
print(f"  Total:          {len(combined_ds)}")

# Verify samples load cleanly
for name, ds in [("NCT QA",nct_qa_ds),("NCT Report",nct_report_ds),("PathVQA",pvqa_ds)]:
    s = ds[0]
    shapes = {k: tuple(v.shape) for k,v in s.items() if hasattr(v,'shape')}
    print(f"  {name}: {shapes}")

print(f"\nVRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")
_, used, free = shutil.disk_usage("/kaggle/working")
print(f"/kaggle/working — Used: {used/1e9:.1f} GB | Free: {free/1e9:.1f} GB")
print("✓ Dataset ready")

Loading NCT-CRC-HE-100K...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/31 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/31 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/31 [00:00<?, ?it/s]

...

Generating train split:   0%|          | 0/19654 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6259 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6719 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


PathVQA: 3000 samples (1500 yn + 1500 open)

Dataset composition:
  NCT-CRC QA:     1800
  NCT-CRC Reports:450
  PathVQA:        3000
  Total:          5250
  NCT QA: {'input_ids': (402,), 'attention_mask': (402,), 'mm_token_type_ids': (402,), 'pixel_values': (2520, 768), 'image_position_ids': (2520, 2)}
  NCT Report: {'input_ids': (461,), 'attention_mask': (461,), 'mm_token_type_ids': (461,), 'pixel_values': (2520, 768), 'image_position_ids': (2520, 2)}
  PathVQA: {'input_ids': (381,), 'attention_mask': (381,), 'mm_token_type_ids': (381,), 'pixel_values': (2520, 768), 'image_position_ids': (2520, 2)}

VRAM: 8.26 GB
/kaggle/working — Used: 0.0 GB | Free: 20.9 GB
✓ Dataset ready


In [7]:
# ============================================================
# CELL 4 — Train (checkpoints → /tmp)
# ============================================================
from trl import SFTTrainer, SFTConfig
from unsloth import is_bf16_supported
from torch.nn.utils.rnn import pad_sequence
import torch, shutil

def collate_fn(batch):
    input_ids      = pad_sequence([b['input_ids'] for b in batch],
                                   batch_first=True,
                                   padding_value=tokenizer.pad_token_id or 0)
    attention_mask = pad_sequence([b['attention_mask'] for b in batch],
                                   batch_first=True, padding_value=0)
    result = {
        'input_ids':      input_ids,
        'attention_mask': attention_mask,
        'labels':         input_ids.clone(),
    }
    if 'pixel_values' in batch[0]:
        result['pixel_values'] = torch.stack([b['pixel_values'] for b in batch])
    return result

FastModel.for_training(model)

for path in ["/kaggle/working", "/tmp"]:
    _, used, free = shutil.disk_usage(path)
    print(f"{path} — Used: {used/1e9:.1f}GB | Free: {free/1e9:.1f}GB")

print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"\nTraining on {len(combined_ds)} samples — 600 steps...")

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = combined_ds,
    data_collator = collate_fn,
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps                = 50,
        max_steps                   = 600,
        learning_rate               = 2e-4,
        fp16                        = not is_bf16_supported(),
        bf16                        = is_bf16_supported(),
        logging_steps               = 50,
        output_dir                  = "/tmp/pathos-train-tmp",  # checkpoints → /tmp
        report_to                   = "none",
        save_strategy               = "no",    # no intermediate saves
        dataloader_num_workers      = 0,
        remove_unused_columns       = False,
        dataset_text_field          = "",
        dataset_kwargs              = {"skip_prepare_dataset": True},
    ),
)

stats = trainer.train()

print(f"\n{'='*50}")
print(f"PATHOS TRAINING COMPLETE")
print(f"{'='*50}")
print(f"Steps: {stats.global_step}")
print(f"Loss:  {stats.training_loss:.4f}")
print(f"Time:  {stats.metrics['train_runtime']/60:.1f} mins")

# Clean trainer temp immediately
shutil.rmtree("/tmp/pathos-train-tmp", ignore_errors=True)

for path in ["/kaggle/working", "/tmp"]:
    _, used, free = shutil.disk_usage(path)
    print(f"{path} — Used: {used/1e9:.1f}GB | Free: {free/1e9:.1f}GB")

/kaggle/working — Used: 0.0GB | Free: 20.9GB
/tmp — Used: 7420.5GB | Free: 1236.4GB
VRAM: 8.26 GB

Training on 5250 samples — 600 steps...


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,250 | Num Epochs = 1 | Total steps = 600
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 29,859,840 of 5,153,037,856 (0.58% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
50,2.987653
100,0.124754
150,0.039285
200,0.028131
250,0.020197
300,0.017119
350,0.015873
400,0.019796
450,0.018486
500,0.014575


/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))



PATHOS TRAINING COMPLETE
Steps: 600
Loss:  0.2762
Time:  82.8 mins
/kaggle/working — Used: 0.0GB | Free: 20.9GB
/tmp — Used: 7420.6GB | Free: 1236.3GB


In [8]:
# CELL 5 — Lightweight save only, no merge, no OOM
import os, shutil

ADAPTER_TMP  = "/tmp/pathos-adapter"
KAGGLE_OUT   = "/kaggle/working/pathos-adapter"

# ── 1. Save adapter only (~150MB, safe) ──────────────────────
print("Saving LoRA adapter → /tmp...")
model.save_pretrained(ADAPTER_TMP)
tokenizer.save_pretrained(ADAPTER_TMP)

files = os.listdir(ADAPTER_TMP)
total_mb = sum(os.path.getsize(os.path.join(ADAPTER_TMP,f)) for f in files)/1e6
print(f"✓ Adapter saved: {total_mb:.0f} MB")
print(f"  Files: {files}")

# ── 2. Copy to /kaggle/working so it appears in output tab ───
if os.path.exists(KAGGLE_OUT):
    shutil.rmtree(KAGGLE_OUT)
shutil.copytree(ADAPTER_TMP, KAGGLE_OUT)
print(f"✓ Copied → {KAGGLE_OUT}")

# ── 3. Disk check ────────────────────────────────────────────
for path in ["/kaggle/working", "/tmp"]:
    _, used, free = shutil.disk_usage(path)
    print(f"{path} — Used: {used/1e9:.1f}GB | Free: {free/1e9:.1f}GB")

print("\n✓ Adapter saved! GGUF will be generated via HuggingFace post-submission.")
print("\nNext step: push adapter to HuggingFace Hub for public weights.")

Saving LoRA adapter → /tmp...


Unsloth: Restored added_tokens_decoder metadata in /tmp/pathos-adapter/tokenizer_config.json.


✓ Adapter saved: 152 MB
  Files: ['tokenizer.json', 'adapter_config.json', 'tokenizer_config.json', 'chat_template.jinja', 'README.md', 'processor_config.json', 'adapter_model.safetensors']
✓ Copied → /kaggle/working/pathos-adapter
/kaggle/working — Used: 0.2GB | Free: 20.8GB
/tmp — Used: 7420.7GB | Free: 1236.2GB

✓ Adapter saved! GGUF will be generated via HuggingFace post-submission.

Next step: push adapter to HuggingFace Hub for public weights.


In [10]:
# CELL 5b — Push to HuggingFace Hub
from huggingface_hub import login
import os

# Add HF_TOKEN to Kaggle secrets:
# Kaggle → Add-ons → Secrets → New Secret → name: HF_TOKEN
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
login(token=hf_token)

HF_REPO = "dhairyapandya/pathos-gemma4-histopathology"

print(f"Pushing adapter to {HF_REPO}...")
model.push_to_hub(HF_REPO, token=hf_token)
tokenizer.push_to_hub(HF_REPO, token=hf_token)

print(f"\n✓ PathOS published!")
print(f"  https://huggingface.co/{HF_REPO}")
print(f"\nGGUF conversion: visit https://huggingface.co/spaces/ggml-org/gguf-my-repo")
print(f"Enter your repo name and it generates Q4_K_M GGUF automatically.")

Pushing adapter to dhairyapandya/pathos-gemma4-histopathology...


README.md:   0%|          | 0.00/575 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/dhairyapandya/pathos-gemma4-histopathology


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmphl2o_5qr/tokenizer_config.json.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


✓ PathOS published!
  https://huggingface.co/dhairyapandya/pathos-gemma4-histopathology

GGUF conversion: visit https://huggingface.co/spaces/ggml-org/gguf-my-repo
Enter your repo name and it generates Q4_K_M GGUF automatically.


In [11]:
# ============================================================
# CELL 6 — Benchmark on 100 validation samples
# ============================================================
from datasets import load_dataset
from unsloth import FastModel
import torch, random, shutil

FastModel.for_inference(model)

pvqa_val = load_dataset(
    "flaviagiammarino/path-vqa",
    cache_dir = "/tmp/hf_datasets_cache"
)['validation']

random.seed(0)
val_yn   = [i for i in range(len(pvqa_val))
            if pvqa_val[i]['answer'].lower() in ['yes','no']][:50]
val_open = [i for i in range(len(pvqa_val))
            if pvqa_val[i]['answer'].lower() not in ['yes','no']][:50]
val_all  = val_yn + val_open
random.shuffle(val_all)

def run_inference(sample, max_new=100):
    img = sample['image']
    if img.mode != 'RGB': img = img.convert('RGB')
    img = img.resize((224, 224))

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": img},
            {"type": "text",
             "text": f"{SYSTEM_PROMPT}\n\nQuestion: {sample['question']}"}
        ]
    }]

    text   = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(
        text=text, images=[img],
        return_tensors="pt", truncation=True, max_length=1024
    ).to("cuda")

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens = max_new,
            temperature    = 0.05,
            do_sample      = True,
        )

    new = out[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new, skip_special_tokens=True).strip()

print("Running 100-sample benchmark...")
correct_yn, total_yn   = 0, 0
match_open, total_open = 0, 0
results = []

for i, idx in enumerate(val_all):
    sample  = pvqa_val[idx]
    gt      = sample['answer'].strip().lower()
    pred    = run_inference(sample).lower()
    is_yn   = gt in ['yes','no']

    if is_yn:
        total_yn += 1
        correct   = pred.startswith(gt)
        if correct: correct_yn += 1
    else:
        total_open += 1
        gt_words = [w for w in gt.split() if len(w) > 3]
        correct  = any(w in pred for w in gt_words) if gt_words else gt in pred
        if correct: match_open += 1

    results.append({
        "q": sample['question'], "gt": gt,
        "pred": pred[:100],
        "type": "yn" if is_yn else "open",
        "correct": correct
    })

    if (i+1) % 25 == 0:
        print(f"  {i+1}/100 evaluated...")

print(f"\n{'='*50}")
print(f"  PATHOS BENCHMARK RESULTS")
print(f"{'='*50}")
print(f"  Yes/No accuracy:  {correct_yn}/{total_yn} ({correct_yn/total_yn*100:.1f}%)")
print(f"  Open-ended match: {match_open}/{total_open} ({match_open/total_open*100:.1f}%)")
print(f"{'='*50}")

print("\nSample predictions:")
for r in results[:8]:
    s = "✓" if r['correct'] else "✗"
    print(f"\n  [{r['type'].upper()}] {s}")
    print(f"  Q:  {r['q']}")
    print(f"  GT: {r['gt']}")
    print(f"  P:  {r['pred'][:90]}")

Running 100-sample benchmark...
  25/100 evaluated...
  50/100 evaluated...
  75/100 evaluated...
  100/100 evaluated...

  PATHOS BENCHMARK RESULTS
  Yes/No accuracy:  16/50 (32.0%)
  Open-ended match: 11/50 (22.0%)

Sample predictions:

  [YN] ✗
  Q:  did nuclear pleomorphism emphasize intimal smooth muscle cell migration and proliferation associated with extracellular matrix synthesis?
  GT: no
  P:  based on the provided image, which appears to be a micrograph of a vascular structure (lik

  [YN] ✓
  Q:  is abdomen present?
  GT: yes
  P:  yes, the image displays a section of the abdomen.

  [YN] ✗
  Q:  does omphalocele show a photo taken during life large lesion?
  GT: no
  P:  yes, the image appears to show an omphalocele, which is a congenital defect where abdomina

  [YN] ✗
  Q:  does this image show bowel in situ with diffuse thickening of peritoneal surfaces due to metastatic carcinoma breast primary i think?
  GT: yes
  P:  based on the provided image, i cannot confirm the 

In [13]:
# ============================================================
# CELL 7 — Live demo: QA + Report generation
# ============================================================
from unsloth import FastModel
import torch

FastModel.for_inference(model)

pvqa_val = load_dataset(
    "flaviagiammarino/path-vqa",
    cache_dir="/tmp/hf_datasets_cache"
)['validation']

for idx in [10, 42, 200]:
    sample = pvqa_val[idx]
    img    = sample['image']
    if img.mode != 'RGB': img = img.convert('RGB')
    img    = img.resize((224, 224))

    print(f"\n{'='*60}")
    print(f"SAMPLE {idx}")
    print(f"GT Question: {sample['question']}")
    print(f"GT Answer:   {sample['answer']}")

    # ── QA inference ────────────────────────────────────────
    msgs = [{"role":"user","content":[
        {"type":"image","image":img},
        {"type":"text","text":f"{SYSTEM_PROMPT}\n\nQuestion: {sample['question']}"}
    ]}]
    text   = tokenizer.apply_chat_template(msgs, tokenize=False,
                                            add_generation_prompt=True)
    inputs = tokenizer(text=text, images=[img], return_tensors="pt",
                       truncation=True, max_length=1024).to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=80,
                              temperature=0.1, do_sample=True)
    ans = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:],
                           skip_special_tokens=True).strip()
    print(f"\nPathOS Answer:\n{ans}")

    # ── Report generation ────────────────────────────────────
    msgs2 = [{"role":"user","content":[
        {"type":"image","image":img},
        {"type":"text","text":f"{SYSTEM_PROMPT}\n\nGenerate a PathOS lab report for this histopathology patch."}
    ]}]
    text2   = tokenizer.apply_chat_template(msgs2, tokenize=False,
                                             add_generation_prompt=True)
    inputs2 = tokenizer(text=text2, images=[img], return_tensors="pt",
                        truncation=True, max_length=1024).to("cuda")
    with torch.no_grad():
        out2 = model.generate(**inputs2, max_new_tokens=280,
                               temperature=0.2, do_sample=True)
    report = tokenizer.decode(out2[0][inputs2['input_ids'].shape[1]:],
                              skip_special_tokens=True).strip()
    print(f"\nPathOS Lab Report:\n{report}")
    print()


SAMPLE 10
GT Question: where is this area in the body?
GT Answer:   abdomen

PathOS Answer:
This image appears to be a cross-section of the gastrointestinal tract, likely the stomach or a portion of the small intestine, given the mucosal folds and submucosal layers visible.

PathOS Lab Report:
**Pathology Report**

**Overall Impression:** Grossly enlarged, lobulated organ with a uniform, granular, reddish-brown surface.

**Detailed Description:**
The image displays a cross-section or surface view of a large, smooth, lobulated organ. The surface is distinctly hyperemic or congested, appearing reddish-brown, which is characteristic of a highly vascularized surface or mucosal lining. The underlying tissue appears relatively uniform and granular in texture. The overall morphology is suggestive of a grossly enlarged visceral organ.

**Differential Diagnosis (Based on Morphology):**
Given the appearance of a large, smooth, highly vascularized surface, differential diagnoses could include:
1

In [14]:
# ============================================================
# CELL 6 — Benchmark on 100 validation samples + Save Results
# ============================================================
from datasets import load_dataset
from unsloth import FastModel
import torch, random, csv, os
import pandas as pd

FastModel.for_inference(model)

pvqa_val = load_dataset(
    "flaviagiammarino/path-vqa",
    cache_dir="/tmp/hf_datasets_cache"
)['validation']

random.seed(0)
val_yn   = [i for i in range(len(pvqa_val))
            if pvqa_val[i]['answer'].lower() in ['yes', 'no']][:50]
val_open = [i for i in range(len(pvqa_val))
            if pvqa_val[i]['answer'].lower() not in ['yes', 'no']][:50]
val_all  = val_yn + val_open
random.shuffle(val_all)

# ── inference ────────────────────────────────────────────────
def run_inference(sample, max_new=200):
    img = sample['image']
    if img.mode != 'RGB':
        img = img.convert('RGB')
    img = img.resize((224, 224))
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": img},
            {"type": "text",
             "text": f"{SYSTEM_PROMPT}\n\nQuestion: {sample['question']}"}
        ]
    }]
    text   = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(
        text=text, images=[img],
        return_tensors="pt", truncation=True, max_length=1024
    ).to("cuda")
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new,
            temperature=0.05,
            do_sample=True,
        )
    new = out[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new, skip_special_tokens=True).strip()

# ── run inference & collect raw results ──────────────────────
print("Running 100-sample benchmark...")
raw_results = []

for i, idx in enumerate(val_all):
    sample = pvqa_val[idx]
    gt     = sample['answer'].strip().lower()
    pred   = run_inference(sample)          # keep original casing for saving
    is_yn  = gt in ['yes', 'no']

    raw_results.append({
        "idx":      idx,
        "type":     "yn" if is_yn else "open",
        "question": sample['question'],
        "gt":       gt,
        "full_prediction": pred,            # full untruncated model output
    })

    if (i + 1) % 25 == 0:
        print(f"  {i+1}/100 evaluated...")

# ── save raw results to CSV ───────────────────────────────────
OUT_DIR  = "/kaggle/working"
RAW_CSV  = os.path.join(OUT_DIR, "pathos_raw_results.csv")

with open(RAW_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["idx","type","question","gt","full_prediction"])
    writer.writeheader()
    writer.writerows(raw_results)

print(f"\n✓ Raw predictions saved → {RAW_CSV}")

# ── smarter evaluation ────────────────────────────────────────
# For yes/no: look for the word anywhere in the first sentence of the answer,
# not just startswith — because Gemma reasons before answering.
# For open: check lemma-level overlap on content words (len > 3).

import re

def extract_final_answer(text):
    """
    Gemma tends to reason then conclude.
    Try to grab the last short sentence or anything after
    'therefore', 'thus', 'answer is', 'conclusion', 'final answer'.
    Falls back to the full text if nothing found.
    """
    text = text.strip()
    patterns = [
        r'(?:the answer is|final answer[:\s]+|answer[:\s]+|therefore[,\s]+|thus[,\s]+|in conclusion[,\s]+)(.*?)(?:\.|$)',
        r'(?:\n|^)([^\n]{1,80})$',   # last short line
    ]
    for p in patterns:
        m = re.search(p, text, re.IGNORECASE | re.DOTALL)
        if m:
            candidate = m.group(1).strip().lower()
            if candidate:
                return candidate
    return text.lower()

def evaluate_yn(gt, full_pred):
    """Return True if the correct yes/no word appears in extracted answer."""
    extracted = extract_final_answer(full_pred)
    opposite  = 'no' if gt == 'yes' else 'yes'
    # Prefer extracted; fall back to full text
    for text in [extracted, full_pred.lower()]:
        words = re.findall(r'\b\w+\b', text)
        if gt in words:
            # make sure opposite isn't mentioned BEFORE gt
            if opposite not in words[:words.index(gt)]:
                return True
    return False

def evaluate_open(gt, full_pred):
    """Overlap on content words (len > 3) between GT and extracted answer."""
    extracted = extract_final_answer(full_pred)
    gt_words  = [w for w in gt.split() if len(w) > 3]
    if not gt_words:
        return gt in extracted
    return any(w in extracted for w in gt_words)

# ── score ─────────────────────────────────────────────────────
correct_yn, total_yn   = 0, 0
match_open, total_open = 0, 0
scored_results = []

for r in raw_results:
    gt   = r['gt']
    pred = r['full_prediction']
    is_yn = r['type'] == 'yn'

    if is_yn:
        total_yn += 1
        correct   = evaluate_yn(gt, pred)
        if correct: correct_yn += 1
    else:
        total_open += 1
        correct    = evaluate_open(gt, pred)
        if correct: match_open += 1

    scored_results.append({**r, "correct": correct,
                            "extracted_answer": extract_final_answer(pred)})

# ── save scored CSV ───────────────────────────────────────────
SCORED_CSV = os.path.join(OUT_DIR, "pathos_scored_results.csv")

df = pd.DataFrame(scored_results)
df.to_csv(SCORED_CSV, index=False)
print(f"✓ Scored results saved  → {SCORED_CSV}")

# ── final report ──────────────────────────────────────────────
print(f"\n{'='*55}")
print(f"  PATHOS BENCHMARK RESULTS  (smarter eval)")
print(f"{'='*55}")
print(f"  Yes/No accuracy:  {correct_yn}/{total_yn} "
      f"({correct_yn/total_yn*100:.1f}%)")
print(f"  Open-ended match: {match_open}/{total_open} "
      f"({match_open/total_open*100:.1f}%)")
overall = (correct_yn + match_open) / 100
print(f"  Overall:          {int(overall*100)}/100 ({overall*100:.1f}%)")
print(f"{'='*55}")

# ── sample predictions ────────────────────────────────────────
print("\nSample predictions (with extracted answer):\n")
for r in scored_results[:8]:
    s = "✓" if r['correct'] else "✗"
    print(f"  [{r['type'].upper()}] {s}")
    print(f"  Q:         {r['question']}")
    print(f"  GT:        {r['gt']}")
    print(f"  Extracted: {r['extracted_answer'][:120]}")
    print(f"  Full pred: {r['full_prediction'][:120]}")
    print()

/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Running 100-sample benchmark...
  25/100 evaluated...
  50/100 evaluated...
  75/100 evaluated...
  100/100 evaluated...

✓ Raw predictions saved → /kaggle/working/pathos_raw_results.csv
✓ Scored results saved  → /kaggle/working/pathos_scored_results.csv

  PATHOS BENCHMARK RESULTS  (smarter eval)
  Yes/No accuracy:  16/50 (32.0%)
  Open-ended match: 10/50 (20.0%)
  Overall:          26/100 (26.0%)

Sample predictions (with extracted answer):

  [YN] ✗
  Q:         did nuclear pleomorphism emphasize intimal smooth muscle cell migration and proliferation associated with extracellular matrix synthesis?
  GT:        no
  Extracted: your question accurately, i would need a specific clinical context or a different image that highlights the features you
  Full pred: Based on the provided image, which appears to be a micrograph of a vascular structure (likely an artery or vein), I cann

  [YN] ✓
  Q:         is abdomen present?
  GT:        yes
  Extracted: yes, the image displays a section o